# 第四轮模型融合 · YOLO26n

沿用第三轮的两类数据划分和 100 轮训练协议。先跑 F01–F06，得到两套双模块模型及四个单模块对照。余下三个名额用于交叉组合 F07、F08 和原版 P5 ViT 单独对照 F09。

选择 **Python (YOLO26 Fusion)** 内核，从项目根目录打开本文件。“全部运行”会构建模型、核对分支并读取已有结果。正式训练命令放在各节的 PowerShell 代码块中，复制到终端后才会训练。

所有终端先进入各自的项目根目录。三台电脑都需要同一套项目代码、训练环境和数据。如果缺少官方 n 权重，只在该机器运行一次下载命令。

```powershell
Set-Location D:\yolo26-vit
& .\.venv-cuda\Scripts\python.exe train.py --download-weights yolo26n
```

## 1 · 数据、权重与原始数据流

固定 622 张训练图、93 张验证图，输入 640×640，batch 4，AdamW，FP32，seed 0，100 轮。第三轮基线从 `baseline_resume01/weights/best.pt` 的复评读取。下方所有层号从 0 开始，`-1` 表示上一层。最后一层列出解码后的检测输出。

In [1]:
from pathlib import Path
import os
import torch
import yaml
from IPython.display import Markdown, display

assert Path("train.py").is_file(), "请从项目根目录打开 Notebook"
os.environ["YOLO_OFFLINE"] = "true"
from models.phase4 import CONFIG, COCO, DATA, DATAFILE, OUTPUT, PROJECT, WEIGHTS, latest_result
from models.phase4_layers import ROUTES, definitions, legacy, make_detector, register
from models.phase4_checks import check_group, print_graph
from fusion.data import inspect_data
from fusion.runtime import sha256
from ultralytics import YOLO

register()
torch.set_num_threads(4)
configs = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
assert configs == definitions() and sha256(WEIGHTS) == COCO
data = inspect_data(DATAFILE)
assert data["sha256"] == DATA and [data["counts"][s]["images"] for s in ("train", "val")] == [622, 93]
assert data["names"] == {0: "UGV", 1: "USV"}
print("训练 622 张，验证 93 张；0 = UGV，1 = USV。COCO 权重 SHA-256", COCO)
coco = YOLO(str(WEIGHTS), verbose=False).model.float()
base_cfg = yaml.safe_load((PROJECT / "models/fusion_phase3.yaml").read_text(encoding="utf-8"))["baseline"]
native = make_detector(base_cfg, nc=2, weights=coco).eval()
image = torch.rand((1, 3, 640, 640), generator=torch.Generator().manual_seed(0))
print(f"原版 YOLO26n 两类参数量 {sum(p.numel() for p in native.parameters()):,}")
print("层     输入来源   模块          输出 (N,C,H,W) / Detect (N,框,6)")
print_graph(native, image)

训练 622 张，验证 93 张；0 = UGV，1 = USV。COCO 权重 SHA-256 9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef
Overriding model.yaml nc=80 with nc=2


原版 YOLO26n 两类参数量 2,504,580
层     输入来源   模块          输出 (N,C,H,W) / Detect (N,框,6)


 0            -1  Conv          (1, 16, 320, 320)
 1            -1  Conv          (1, 32, 160, 160)
 2            -1  C3k2          (1, 64, 160, 160)
 3            -1  Conv          (1, 64, 80, 80)
 4            -1  C3k2          (1, 128, 80, 80)
 5            -1  Conv          (1, 128, 40, 40)
 6            -1  C3k2          (1, 128, 40, 40)
 7            -1  Conv          (1, 256, 20, 20)
 8            -1  C3k2          (1, 256, 20, 20)
 9            -1  SPPF          (1, 256, 20, 20)
10            -1  C2PSA         (1, 256, 20, 20)
11            -1  Upsample      (1, 256, 40, 40)
12       [-1, 6]  Concat        (1, 384, 40, 40)
13            -1  C3k2          (1, 128, 40, 40)
14            -1  Upsample      (1, 128, 80, 80)
15       [-1, 4]  Concat        (1, 256, 80, 80)
16            -1  C3k2          (1, 64, 80, 80)
17            -1  Conv          (1, 64, 40, 40)
18      [-1, 13]  Concat        (1, 192, 40, 40)
19            -1  C3k2          (1, 128, 40, 40)
20            -1  Co

P3、P4、P5 分别取自 L16、L19、L22，空间尺寸依次为 80×80、40×40、20×20。L10 同时通往 L11 上采样和 L21 拼接，L13 同时通往 L14 上采样和 L18 拼接。L19 同时通往 L20 下采样和 L23 检测头。

以下 Mamba 复用项目的四方向 SS2D，在 20×20 上每个方向扫描 400 个位置，宽度 32、状态维度 8。轻量 ViT 把 128×40×40 池化为 10×10，投影至 64 维后做一层 4 头注意力，前馈宽度 128，再上采样回原尺寸。两个分支均采用零初始化的可学习残差门控。F09 使用第三轮的原版模块。

每节先用官方权重构建 80 类模型，再按训练器同样的方式重建两类模型。验收时核对可迁移张量、对比零门控整组输出，并把门控临时设为 0.25 检查实际接线和三路检测头尺寸。

## F01 · 末端 Mamba

**接线**　L22 → Mamba → L23[P5]。

与基线比较，检验扫描最终 P5 的收益。与 F04 比较拼接前后的差异。

In [2]:
wrapper, transfer = legacy.build_model("F01", yolo_weights=WEIGHTS, architecture=configs["F01"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F01", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,525,477；COCO 原始张量已核对 708 项
80 → 2 类迁移 620/722 项；零门控输出与基线逐元素完全一致
L22 → L23   (1, 256, 20, 20) → (1, 256, 20, 20)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F01**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F01 --device 0
```

## F02 · 检测头 ViT

**接线**　L19 → 轻量 ViT → L23[P4]；L19 → L20 保留原特征。

与基线比较，定位只修改 P4 检测输入的作用。与 F05 比较 ViT 接法。

In [3]:
wrapper, transfer = legacy.build_model("F02", yolo_weights=WEIGHTS, architecture=configs["F02"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F02", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,554,629；COCO 原始张量已核对 708 项
80 → 2 类迁移 623/725 项；零门控输出与基线逐元素完全一致
L19 → L23   (1, 128, 40, 40) → (1, 128, 40, 40)
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F02**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F02 --device 0
```

## F03 · 末端 Mamba ＋ 检测头 ViT

**接线**　L22 → Mamba → L23[P5]；L19 → 轻量 ViT → L23[P4]。

与 F01、F02 分别比较新增另一分支的收益，再与基线比较。两路特征第一次相遇在检测头。

In [4]:
wrapper, transfer = legacy.build_model("F03", yolo_weights=WEIGHTS, architecture=configs["F03"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F03", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,575,526；COCO 原始张量已核对 708 项
80 → 2 类迁移 637/739 项；零门控输出与基线逐元素完全一致
L22 → L23   (1, 256, 20, 20) → (1, 256, 20, 20)
L19 → L23   (1, 128, 40, 40) → (1, 128, 40, 40)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F03**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F03 --device 0
```

## F04 · 跳连 Mamba

**接线**　L10 → Mamba → L21[跳连]；L10 → L11 保留原特征。

与基线和 F01 比较。F01、F04 的扫描尺寸和分支宽度相同，输入特征的语义和接入位置不同。

In [5]:
wrapper, transfer = legacy.build_model("F04", yolo_weights=WEIGHTS, architecture=configs["F04"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F04", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,525,477；COCO 原始张量已核对 708 项
80 → 2 类迁移 620/722 项；零门控输出与基线逐元素完全一致
L10 → L21   (1, 256, 20, 20) → (1, 256, 20, 20)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F04**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F04 --device 0
```

## F05 · 跳连 ViT

**接线**　L13 → 轻量 ViT → L18[跳连]；L13 → L14 保留原特征。

与基线和 F02 比较。该跳连影响下游 P4 与 P5，通往 P3 的上采样路径保留。

In [6]:
wrapper, transfer = legacy.build_model("F05", yolo_weights=WEIGHTS, architecture=configs["F05"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F05", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,554,629；COCO 原始张量已核对 708 项
80 → 2 类迁移 623/725 项；零门控输出与基线逐元素完全一致
L13 → L18   (1, 128, 40, 40) → (1, 128, 40, 40)
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F05**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F05 --device 0
```

## F06 · 跳连 Mamba ＋ 跳连 ViT

**接线**　L10 → Mamba → L21[跳连]；L13 → 轻量 ViT → L18[跳连]。

与 F04、F05 比较两支联合使用的增益。与 F03 比较整套接法，两处同时变化。

In [7]:
wrapper, transfer = legacy.build_model("F06", yolo_weights=WEIGHTS, architecture=configs["F06"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F06", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,575,526；COCO 原始张量已核对 708 项
80 → 2 类迁移 637/739 项；零门控输出与基线逐元素完全一致
L10 → L21   (1, 256, 20, 20) → (1, 256, 20, 20)
L13 → L18   (1, 128, 40, 40) → (1, 128, 40, 40)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F06**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F06 --device 0
```

## F07 · 末端 Mamba ＋ 跳连 ViT

**接线**　L22 → Mamba → L23[P5]；L13 → 轻量 ViT → L18[跳连]。

与 F03 只差 ViT 位置，与 F06 只差 Mamba 位置。

In [8]:
wrapper, transfer = legacy.build_model("F07", yolo_weights=WEIGHTS, architecture=configs["F07"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F07", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,575,526；COCO 原始张量已核对 708 项
80 → 2 类迁移 637/739 项；零门控输出与基线逐元素完全一致
L22 → L23   (1, 256, 20, 20) → (1, 256, 20, 20)
L13 → L18   (1, 128, 40, 40) → (1, 128, 40, 40)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F07**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F07 --device 0
```

## F08 · 跳连 Mamba ＋ 检测头 ViT

**接线**　L10 → Mamba → L21[跳连]；L19 → 轻量 ViT → L23[P4]。

与 F03 只差 Mamba 位置，与 F06 只差 ViT 位置，补齐两种位置的组合。

In [9]:
wrapper, transfer = legacy.build_model("F08", yolo_weights=WEIGHTS, architecture=configs["F08"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F08", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,575,526；COCO 原始张量已核对 708 项
80 → 2 类迁移 637/739 项；零门控输出与基线逐元素完全一致
L10 → L21   (1, 256, 20, 20) → (1, 256, 20, 20)
L19 → L23   (1, 128, 40, 40) → (1, 128, 40, 40)
ResidualSS2D  四方向 × 400 个位置，宽度 32，状态维度 8
PooledViT  注意力输入 (1, 100, 64)，4 头，前馈宽度 128
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F08**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F08 --device 0
```

## F09 · 第三轮原版 P5 ViT

**接线**　L10 → 原版 P5 ViT → L11 与 L21；无 Mamba。

沿用第三轮 C 的 P5ResidualViT，20×20 共 400 个位置，维度 128、4 头、前馈宽度 512。和第三轮基线比较 ViT 单独的效应，再结合第三轮 B、C 比较与 P4 Mamba 的交互。

In [10]:
wrapper, transfer = legacy.build_model("F09", yolo_weights=WEIGHTS, architecture=configs["F09"])
assert transfer["yolo"]["sha256"] == COCO
detector = make_detector(wrapper.model.yaml, nc=2, weights=wrapper.model).eval()
check_group("F09", wrapper.model, detector, native, image, coco)

Overriding model.yaml nc=80 with nc=2


两类参数 2,768,773；COCO 原始张量已核对 708 项
80 → 2 类迁移 623/725 项；零门控输出与基线逐元素完全一致
L10 原版 P5 ViT   (1, 256, 20, 20) → (1, 256, 20, 20)，两条下游路径均读取其输出
P5ResidualViT  注意力输入 (1, 400, 128)，4 头，前馈宽度 512
Detect 实收 P3 / P4 / P5 [(64, 80, 80), (128, 40, 40), (256, 20, 20)]


**单独训练 F09**　从项目根目录运行。若曾中断，重新执行同一条命令即可查找该组的有效检查点。

```powershell
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F09 --device 0
```

## 三张显卡的训练队列

三个终端各执行自己的一行，每个进程依次完成分配给它的模型。三台单卡电脑均用 GPU 0；一台三卡电脑按表中物理卡编号运行。每组使用独立的 `phase4_n/Fxx/attempt_XX`，保存控制台日志、`weights/best.pt`、`weights/last.pt`、续训来源和最终验证指标。中断后在原环境重发该命令；有已保存的完整轮次就从下一轮恢复，首轮前中断会重新初始化。已完成且复评通过的组会跳过。

| 机器 / GPU | 第一批 | 第二批 | 可选第三批 |
|---|---|---|---|
| A / 0 | F01 | F06 | F09 |
| B / 1 | F02 | F04 | F07 |
| C / 2 | F03 | F05 | F08 |

**先跑六组 · 三台单卡电脑**

```powershell
## 电脑 A
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F01 F06 --device 0
## 电脑 B
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F02 F04 --device 0
## 电脑 C
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F03 F05 --device 0
```

**先跑六组 · 同机三卡，三个独立终端**

```powershell
## 终端 A
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F01 F06 --device 0
## 终端 B
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F02 F04 --device 1
## 终端 C
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F03 F05 --device 2
```

**决定做满九组时**，以下每条都支持从头运行或在前六组完成后补跑。若使用三台电脑，把 B、C 行的设备号也设为 0。

```powershell
## A
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F01 F06 F09 --device 0
## B
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F02 F04 F07 --device 1
## C
& .\.venv-cuda\Scripts\python.exe -m models.phase4 F03 F05 F08 --device 2
```

## 用 best.pt 的独立复评汇总

从各组的所有训练尝试中查找完成 100 轮的目录，核对数据、协议、权重哈希及复评设置。多台电脑训练时，将各机 `phase4_n` 下对应的 **Fxx 文件夹整体**复制到汇总机的同名目录，再运行下一格。表中 mAP 均为百分比。初筛时看双模块整体 mAP50 能否达到基线的 84.31%、mAP50–95 能否高于 64.35%，同时核对 UGV 和 USV 各自的变化。93 张验证图且只用一个 seed，微小差距仅供筛选。

In [11]:
print("汇总根目录", OUTPUT)
rows = ["| 组别 | 状态 | UGV mAP50 | UGV mAP50–95 | USV mAP50 | USV mAP50–95 | 整体 mAP50 | 整体 mAP50–95 | 来源 |",
        "|---|---|---:|---:|---:|---:|---:|---:|---|"]
for key in ("baseline", *ROUTES):
    status, path, metrics = latest_result(key)
    values = ["—"] * 6
    if metrics:
        groups = [metrics["per_class"]["UGV"], metrics["per_class"]["USV"], metrics["overall"]]
        values = [f"{group[m]:.2%}" for group in groups for m in ("map50", "map50_95")]
    origin = path.relative_to(OUTPUT.parent).as_posix() if path else "—"
    rows.append("| " + " | ".join([key, status, *values, origin]) + " |")
display(Markdown("\n".join(rows)))

汇总根目录 D:\yolo26-vit\training\artifacts\studies\760e212a7ba42aaf20b2f912aaa688c72c01bbc6c4eacd7af771866161605e8b\phase4_n


| 组别 | 状态 | UGV mAP50 | UGV mAP50–95 | USV mAP50 | USV mAP50–95 | 整体 mAP50 | 整体 mAP50–95 | 来源 |
|---|---|---:|---:|---:|---:|---:|---:|---|
| baseline | 已完成 | 88.48% | 70.11% | 80.14% | 58.59% | 84.31% | 64.35% | phase3_n/baseline_resume01 |
| F01 | 待训练 | — | — | — | — | — | — | — |
| F02 | 待训练 | — | — | — | — | — | — | — |
| F03 | 待训练 | — | — | — | — | — | — | — |
| F04 | 待训练 | — | — | — | — | — | — | — |
| F05 | 待训练 | — | — | — | — | — | — | — |
| F06 | 待训练 | — | — | — | — | — | — | — |
| F07 | 待训练 | — | — | — | — | — | — | — |
| F08 | 待训练 | — | — | — | — | — | — | — |
| F09 | 待训练 | — | — | — | — | — | — | — |